# Analisis Matching SIFT

Notebook ini mendemonstrasikan feature matching SIFT antara satu citra original dan tiga versi augmentasi: Gaussian blur, Gaussian noise, dan JPEG compression. Alurnya mengikuti `src/matching.py` dan `src/visualize_matching.py`, tetapi disusun per langkah agar cocok untuk demo live final project Computer Vision.

## 1. Import Library dan Definisi Path

Kita menggunakan `pathlib` untuk path project, OpenCV untuk SIFT dan matching, pandas untuk tabel ringkasan, serta matplotlib untuk visualisasi.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ORIGINAL_DIR = PROJECT_ROOT / "dataset" / "original" / "train" / "real"
AUGMENTATIONS = {
    "gaussian_blur": PROJECT_ROOT / "dataset" / "gaussian_blur" / "train" / "real",
    "gaussian_noise": PROJECT_ROOT / "dataset" / "gaussian_noise" / "train" / "real",
    "jpeg_compression": PROJECT_ROOT / "dataset" / "jpeg_compression" / "train" / "real",
}
RATIO_THRESHOLD = 0.75

plt.rcParams["figure.figsize"] = (12, 5)

## 2. Fungsi Helper

Fungsi helper berikut digunakan agar notebook tetap rapi. Fungsi-fungsi ini memilih JPG original pertama, mencari citra augmentasi yang sesuai, memuat citra dengan OpenCV, dan mengubah format BGR ke RGB untuk ditampilkan dengan matplotlib.

In [ ]:
def find_first_jpg(folder: Path) -> Path:
    """Mengambil citra JPG pertama dari sebuah folder berdasarkan urutan nama file."""
    jpg_paths = sorted(
        {
            path
            for pattern in ("*.jpg", "*.jpeg", "*.JPG", "*.JPEG")
            for path in folder.glob(pattern)
        }
    )
    if not jpg_paths:
        raise FileNotFoundError(f"Tidak ada citra JPG di {folder}")
    return jpg_paths[0]


def get_original_image_id(image_path: Path) -> str:
    """Mengambil id citra yang sama dari nama file original."""
    suffix = "_original"
    if not image_path.stem.endswith(suffix):
        raise ValueError(f"Nama citra original seharusnya diakhiri dengan '{suffix}': {image_path}")
    return image_path.stem[: -len(suffix)]


def find_augmented_image(original_path: Path, augmentation: str, folder: Path) -> Path:
    """Mencari citra augmentasi yang sesuai dengan citra original terpilih."""
    image_id = get_original_image_id(original_path)
    for extension in (".jpg", ".jpeg", ".JPG", ".JPEG"):
        candidate = folder / f"{image_id}_{augmentation}{extension}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Tidak ditemukan citra {augmentation} yang sesuai untuk {original_path.name}")


def load_bgr_image(image_path: Path):
    """Memuat citra dengan OpenCV dalam format BGR."""
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Gagal memuat citra: {image_path}")
    return image


def bgr_to_rgb(image):
    """Mengubah citra BGR dari OpenCV ke RGB untuk matplotlib."""
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

## 3. Memuat Citra Original dan Citra Augmentasi

Demo ini menggunakan JPG original pertama, lalu memuat citra dengan id yang sama dari setiap folder augmentasi. Dengan cara ini, perbandingan lebih adil karena semua citra augmentasi berasal dari frame sumber yang sama.

In [ ]:
original_path = find_first_jpg(ORIGINAL_DIR)
augmented_paths = {
    name: find_augmented_image(original_path, name, folder)
    for name, folder in AUGMENTATIONS.items()
}

original_bgr = load_bgr_image(original_path)
augmented_bgr = {
    name: load_bgr_image(path)
    for name, path in augmented_paths.items()
}

print(f"Citra original: {original_path}")
for name, path in augmented_paths.items():
    print(f"{name}: {path}")

## 4. Menampilkan Citra Original dan Citra Augmentasi

Pemeriksaan visual ini membantu memastikan bahwa file yang dimuat sudah sesuai dan setiap augmentasi mengubah citra dengan karakteristik yang berbeda.

In [ ]:
images_to_show = {"original": original_bgr, **augmented_bgr}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, image) in zip(axes, images_to_show.items()):
    ax.imshow(bgr_to_rgb(image))
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Mengubah Citra ke Grayscale

SIFT bekerja berdasarkan pola intensitas, sehingga citra diubah ke grayscale sebelum proses deteksi keypoint dan ekstraksi descriptor.

In [ ]:
original_gray = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2GRAY)
augmented_gray = {
    name: cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    for name, image in augmented_bgr.items()
}

gray_images_to_show = {"original": original_gray, **augmented_gray}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, image) in zip(axes, gray_images_to_show.items()):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"{title} grayscale")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Menjalankan Deteksi Keypoint SIFT

SIFT mendeteksi keypoint lokal yang relatif stabil dan membuat descriptor yang merangkum pola gradien di sekitar setiap keypoint.

In [ ]:
sift = cv2.SIFT_create()

original_keypoints, original_descriptors = sift.detectAndCompute(original_gray, None)
augmented_features = {}

for name, image_gray in augmented_gray.items():
    keypoints, descriptors = sift.detectAndCompute(image_gray, None)
    augmented_features[name] = {
        "keypoints": keypoints,
        "descriptors": descriptors,
    }

print(f"Keypoint original: {len(original_keypoints)}")
for name, features in augmented_features.items():
    print(f"Keypoint {name}: {len(features['keypoints'])}")

## 7. Menampilkan Keypoint SIFT

Visualisasi keypoint menunjukkan lokasi struktur lokal yang ditemukan oleh SIFT. Lingkaran yang lebih besar menunjukkan skala deteksi yang lebih besar.

In [ ]:
def draw_sift_keypoints(image_bgr, keypoints):
    """Menggambar keypoint SIFT pada citra."""
    return cv2.drawKeypoints(
        image_bgr,
        keypoints,
        None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )


keypoint_images = {
    "original": draw_sift_keypoints(original_bgr, original_keypoints),
}
for name, image in augmented_bgr.items():
    keypoint_images[name] = draw_sift_keypoints(
        image,
        augmented_features[name]["keypoints"],
    )

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, image) in zip(axes, keypoint_images.items()):
    ax.imshow(bgr_to_rgb(image))
    ax.set_title(f"Keypoint {title}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 8. Mencocokkan Descriptor dengan BFMatcher dan Lowe Ratio Test

Brute-force matcher dengan `cv2.NORM_L2` digunakan untuk membandingkan descriptor SIFT. Lowe Ratio Test mempertahankan match hanya jika match terbaik jauh lebih baik dibandingkan match terbaik kedua.

In [ ]:
def get_good_matches(descriptors_a, descriptors_b):
    """Menerapkan BFMatcher dan Lowe Ratio Test."""
    if descriptors_a is None or descriptors_b is None or len(descriptors_b) < 2:
        return []

    matcher = cv2.BFMatcher(cv2.NORM_L2)
    knn_matches = matcher.knnMatch(descriptors_a, descriptors_b, k=2)

    good_matches = []
    for match_pair in knn_matches:
        if len(match_pair) < 2:
            continue

        m, n = match_pair
        if m.distance < RATIO_THRESHOLD * n.distance:
            good_matches.append(m)

    return good_matches


matches_by_augmentation = {}
for name, features in augmented_features.items():
    matches_by_augmentation[name] = get_good_matches(
        original_descriptors,
        features["descriptors"],
    )

summary_df = pd.DataFrame(
    [
        {
            "augmentation": name,
            "keypoints_original": len(original_keypoints),
            "keypoints_augmented": len(augmented_features[name]["keypoints"]),
            "good_matches": len(matches),
        }
        for name, matches in matches_by_augmentation.items()
    ]
)

summary_df

## 9. Menampilkan Visualisasi Matching

Setiap visualisasi menghubungkan good matches SIFT antara citra original dan satu citra augmentasi. Jumlah garis yang lebih sedikit menunjukkan korespondensi descriptor yang lebih sedikit untuk sample ini.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 15))

for ax, (name, matches) in zip(axes, matches_by_augmentation.items()):
    matched_bgr = cv2.drawMatches(
        original_bgr,
        original_keypoints,
        augmented_bgr[name],
        augmented_features[name]["keypoints"],
        matches,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    ax.imshow(bgr_to_rgb(matched_bgr))
    ax.set_title(f"Original vs {name}: {len(matches)} good matches")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 10. Membuat Bar Chart Good Matches

Bar chart ini merangkum jumlah good matches untuk setiap augmentasi, sehingga pengaruh masing-masing distorsi terhadap matching SIFT lebih mudah dibandingkan.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    summary_df["augmentation"],
    summary_df["good_matches"],
    color=["#4C78A8", "#F58518", "#54A24B"],
)

ax.set_title("Jumlah Good Matches SIFT per Augmentasi")
ax.set_xlabel("Augmentasi")
ax.set_ylabel("Good Matches")

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

## Kesimpulan Demo

Untuk sample ini, jumlah good matches menunjukkan seberapa kuat setiap augmentasi mengubah struktur lokal citra. JPEG compression mempertahankan match paling banyak, Gaussian blur menghasilkan jumlah match sedang, dan Gaussian noise menurunkan match paling besar.